In [1]:
import pandas as pd
import numpy as np
import datetime
from datetime import datetime
import warnings
from fractions import Fraction
warnings.filterwarnings('ignore')
from IPython.display import display
from tqdm import tqdm

In [2]:
df = pd.read_excel('0. Dados Brutos.xlsx')  

In [3]:
df_base = df
pd.options.display.max_columns = None

#ajustes de colunas
df_base = df_base.drop([0, 1])
df_base.columns = df_base.iloc[0]
df_base = df_base.reset_index(drop=True)
df_base = df_base.drop([0])
df_base = df_base.reset_index(drop=True)

#filtrando ativas e cortada cavalete
df_base['SIT._LIG_AGUA'] = df_base['SIT._LIG_AGUA'].astype(str)
df_base = df_base.loc[(df_base["SIT._LIG_AGUA"] == "Ativa") |(df_base['SIT._LIG_AGUA'] == "Cortada Cavalete")]
df_base = df_base.reset_index(drop=True)

# Taxas

In [4]:
#taxa comercial ou pública
tarifa10cp = 6.6338
tarifamaxcp = 10.7461

#taxa residencial
tarifa10res = 4.5011
tarifa25res = 8.2953
tarifa50res = 11.5227
tarifamaxres = 14.2304

#taxa industrial
tarifa10ind = 6.681
tarifamaxind = 10.7461 

# Funções

In [5]:
def obter_ano_fabricacao(numero_hidrometro):
    if len(numero_hidrometro) < 3 or not numero_hidrometro[0].isalpha():
        return np.nan
    
    try:
        ano_fabricacao = 2000 + int(numero_hidrometro[1:3])
        ano_atual = datetime.now().year
        if ano_fabricacao > ano_atual:
            return np.nan
        return ano_fabricacao
    except ValueError:
        return np.nan
    
def obter_idade_hd(data_instalacao, ano_instalacao, ano_fabricacao):
    if not pd.isnull(data_instalacao) and not pd.isnull(ano_instalacao):
        hoje = datetime.now()
        idade_em_dias = (hoje - data_instalacao).days
        idade_em_anos = idade_em_dias / 365
        return round(idade_em_anos, 2)
    return np.nan

def classificar_modelo(consumo, limites_inferiores, limites_superiores, tipos):
    for i in range(len(tipos)):
        if limites_inferiores[i] <= consumo < limites_superiores[i]:
            return tipos[i]
    return np.nan

def determinar_criterio_troca(row):
    if semtrocar:
        return 'UCs não classificadas para troca'
    if c1:
        return 'UCs classificadas para troca por idade, somente'
    if c2:
        return 'UCs classificadas para troca por extrapolação do limite de registro, somente'
    if c3:
        return 'UCs classificadas para troca por dimensionamento, somente'
    if c1_2:
        return 'UCs classificadas para troca por idade e extrapolação de registro'
    if c1_3:
        return 'UCs classificadas para troca por idade e dimensionamento'
    if c2_3:
        return 'UCs classificadas para troca por limite de registro e dimensionamento'
    if c1_2_3:
        return 'UCs classificadas pelos três critérios'
    return np.nan

def calcular_fatura(consumo,economias,categoria):
    #olhar valores aqui: 
    #ajustar parte das economias
    valor = 0
    if categoria == 'Residencial':
        consumo = consumo/economias
        if consumo < 10:
            valor = tarifa10res*10
        if consumo>=10 and consumo<25:
            valor = 10*tarifa10res + (consumo-10)*tarifa25res
        if consumo>=25 and consumo<50:
            valor = 10*tarifa10res + 15*tarifa25res + (consumo-25)*tarifa50res
        if consumo >=50:
            valor = 10*tarifa10res + 15*tarifa25res + 25*tarifa50res + (consumo-50)*tarifamaxres
        valor = valor*economias             
    if categoria == 'Comercial' or categoria == 'Pública':
        consumo = consumo/economias 
        if consumo < 10:
            valor = tarifa10cp*10
        if consumo>=10:
            valor = 10*tarifa10cp + (consumo-10)*tarifamaxcp
        valor = valor*economias  
    if categoria == 'Industrial':
        consumo = consumo/economias 
        if consumo < 10:
            valor = tarifa10ind*10
        if consumo>=10:
            valor = 10*tarifa10ind + (consumo-10)*tarifamaxind
        valor = valor*economias 
    return valor

# Base com colunas desejadas

In [6]:
#Inserir o nome das colunas --------------------------------------------------------

matricula = 'MATRICULA'
bairro = 'BAIRRO'
numero_hidrometro = 'NUMERO_HIDROMETRO'
marca_hidrometro = 'MARCA_HIDROMETRO'
categoria_principal = 'CATEGORIA_PRINCIPAL'
data_instalacao_hidrometro = 'DATA_INSTALACAO_HIDROMETRO'
capacidade_hidrometro = 'CAPACIDADE_HIDROMETRO'
diametro_hidrometro = 'DIAMETRO_HIDROMETRO'
economias = ['NUMERO_ECONOMIAS_RES',
             'NUMERO_ECONOMIAS_COM',
             'NUMERO_ECONOMIAS_IND',
             'NUMERO_ECONOMIAS_PUB']
volume_mensal = ['VOLUME_REAL_01',
                 'VOLUME_REAL_02',
                 'VOLUME_REAL_03',
                 'VOLUME_REAL_04',
                 'VOLUME_REAL_05',
                 'VOLUME_REAL_06',                 
                 'VOLUME_REAL_07',
                 'VOLUME_REAL_08',
                 'VOLUME_REAL_09',
                 'VOLUME_REAL_10',
                 'VOLUME_REAL_11',
                 'VOLUME_REAL']


#Cria-se base para análise com colunas desejadas ---------------------
colunas_desejadas = [matricula, bairro, numero_hidrometro, marca_hidrometro, categoria_principal,
                    data_instalacao_hidrometro, capacidade_hidrometro, diametro_hidrometro]
colunas_desejadas.extend(economias)
colunas_desejadas.extend(volume_mensal)

df_analise = df_base[colunas_desejadas]

# Tratamento dos dados

In [7]:
# Transformar capacidade do hidrômetro em número --------------------------------------------
df_analise[capacidade_hidrometro] = df_analise[capacidade_hidrometro].str.replace(',', '.').astype(float)

# Ajuste da coluna de diâmetro --------------------------------------------
df_analise[diametro_hidrometro] = df_analise[diametro_hidrometro].str.rstrip('"')

#Transformar data_instalacao_hidrometro em data
df_analise[data_instalacao_hidrometro] = pd.to_datetime(df_analise[data_instalacao_hidrometro], format = '%d/%m/%Y')

# Transformar capacidade do hidrômetro em número --------------------------------------------
df_analise[numero_hidrometro] = df_analise[numero_hidrometro].astype(str)

## Criação da coluna quantidade de economias------------------------------

df_analise['Quantidade de economias'] = df_analise[economias].sum(axis=1)
df_analise.drop(columns=economias, axis=1, inplace=True)

## Criação da coluna tipos de hidrômetros----------------------------------

df_analise['Tipo de hidrômetro'] = np.where(df_analise[capacidade_hidrometro] < 30, 'Velocimétrico',
                                        np.where(df_analise[capacidade_hidrometro].notnull(), 'Woltmann', np.nan))

## Ano de instalação do HD --------------------------------------------------

df_analise['Ano de instalação'] = df_analise[data_instalacao_hidrometro].dt.year


## Pegar ano de fabricação com função 4.1 ---------------------------------------

df_analise['Ano de fabricação'] = df_analise[numero_hidrometro].apply(obter_ano_fabricacao)


## Pegar ano de fabricação com função 4.2 -------------------------------------------

df_analise['Idade do HD (anos)'] = df_analise.apply(lambda row: obter_idade_hd(row[data_instalacao_hidrometro],
                                                                     row['Ano de instalação'],
                                                                     row['Ano de fabricação']),axis=1)

## Média do consumo -----------------------------------------------------------------

for column in volume_mensal:
    df_analise[column] = np.where(df_analise[column].isna(), np.nan, df_analise[column])
df_analise['Consumo médio (m³/mes)'] = df_analise[volume_mensal].mean(axis=1, skipna=True)
df_analise.drop(columns=volume_mensal, axis=1, inplace=True)

## Classificação por idade -------------------------------------------------------------

df_analise['Classificação 1: Idade'] = np.where(df_analise['Idade do HD (anos)'] >= 5, 'Classificado para troca',
                                             np.where(df_analise['Idade do HD (anos)'] < 5, 'Não classificado para troca',
                                                      np.nan))

## Registro estimado -------------------------------------------------------------

df_analise['Registro estimado (m³)'] = np.where(df_analise['Consumo médio (m³/mes)'] != np.nan,
                                              df_analise['Consumo médio (m³/mes)']*12*df_analise['Idade do HD (anos)'],
                                              np.nan)

## Registro máximo por tipo de hidrômetro -------------------------------------------------

dim_hd = pd.read_excel('0. Dimensionamento de HDs.xlsx')
dim_hd = dim_hd.drop([0])
dim_hd = dim_hd.reset_index(drop=True)

df_analise = df_analise.merge(dim_hd[['Qmáx (m³/h)', 'Limite máximo registro']],
                              left_on=capacidade_hidrometro, right_on='Qmáx (m³/h)', how='left')

df_analise.drop(columns=['Qmáx (m³/h)'], inplace=True)
df_analise.rename(columns={'Limite máximo registro': 'Registro máximo (m³)'}, inplace=True)

# Classificação por limite de registro -------------------------------------------------

df_analise['Classificação 2: Limite de registro'] = np.where(df_analise['Registro estimado (m³)']<=
                                                          df_analise['Registro máximo (m³)'],
                                                          'Não classificado para troca',
                                                          np.where(df_analise['Registro estimado (m³)']>
                                                                   df_analise['Registro máximo (m³)'],
                                                                   'Classificado para troca',
                                                                   np.nan))

## modelo, diametro, capacidade e valor do modelo ideal ---------------------------------------------------

dim_hd = pd.read_excel('0. Dimensionamento de HDs.xlsx')
df_analise['Modelo ideal'] = df_analise['Consumo médio (m³/mes)'].apply(
    lambda x: classificar_modelo(x, dim_hd['Limite inferior de consumo (m³/mês)'].values,
                                     dim_hd['Limite superior de consumo (m³/mês)'].values, dim_hd['Classificação'].values))

dim_hd['Diâmetro (mm)'] = dim_hd['Diâmetro (mm)'].apply(lambda x: str(x))
dim_hd['Diâmetro (mm)'] = dim_hd['Diâmetro (mm)'].apply(lambda x: str(Fraction(x)))

df_analise = df_analise.merge(dim_hd[['Classificação','Qmáx (m³/h)', 'Diâmetro (mm)','Valor investimento']],
                              left_on='Modelo ideal', right_on='Classificação', how='left')
df_analise.drop(columns=['Classificação'], inplace=True)
df_analise.rename(columns={'Qmáx (m³/h)': 'Capacidade ideal (m³/h)',
                           'Diâmetro (mm)': 'Diâmetro ideal (mm)',
                           'Valor investimento':'Custo de troca (R$)'}, inplace=True)

## 5.14. Classificação por modelo --------------------------------------------------------------------

condicao_igual = (df_analise[capacidade_hidrometro] == df_analise['Capacidade ideal (m³/h)']) & (df_analise[diametro_hidrometro] == df_analise['Diâmetro ideal (mm)'])

df_analise['Classificação 3: Dimensionamento'] = np.where(condicao_igual,"Não classificado para troca",
    np.where(df_analise['CAPACIDADE_HIDROMETRO'].notna() & df_analise['DIAMETRO_HIDROMETRO'].notna(),
             "Classificado para troca",
             np.nan)
)

## Classificação para troca: Se algum dos indicadores disse que deve-se trocar, troca-se.--------------

condicao_troca = (df_analise['Classificação 1: Idade'] == 'Classificado para troca') | (df_analise['Classificação 2: Limite de registro'] == 'Classificado para troca') | (df_analise['Classificação 3: Dimensionamento'] == 'Classificado para troca')
condicao_nan = df_analise['Classificação 1: Idade'].isna() | df_analise['Classificação 2: Limite de registro'].isna() | df_analise['Classificação 3: Dimensionamento'].isna()

#quais critérios classificam pra troca? --------------------------------------------------------------------

def determinar_criterio_troca(row):
    semtrocar = (row['Classificação 1: Idade'] == 'Não classificado para troca') and (row['Classificação 2: Limite de registro'] == 'Não classificado para troca') and (row['Classificação 3: Dimensionamento'] == 'Não classificado para troca')
    c1 = (row['Classificação 1: Idade'] == 'Classificado para troca') and (row['Classificação 2: Limite de registro'] == 'Não classificado para troca') and (row['Classificação 3: Dimensionamento'] == 'Não classificado para troca')
    c2 = (row['Classificação 1: Idade'] == 'Não classificado para troca') and (row['Classificação 2: Limite de registro'] == 'Classificado para troca') and (row['Classificação 3: Dimensionamento'] == 'Não classificado para troca')
    c3 = (row['Classificação 1: Idade'] == 'Não classificado para troca') and (row['Classificação 2: Limite de registro'] == 'Não classificado para troca') and (row['Classificação 3: Dimensionamento'] == 'Classificado para troca')
    c1_2 = (row['Classificação 1: Idade'] == 'Classificado para troca') and (row['Classificação 2: Limite de registro'] == 'Classificado para troca') and (row['Classificação 3: Dimensionamento'] == 'Não classificado para troca')
    c1_3 = (row['Classificação 1: Idade'] == 'Classificado para troca') and (row['Classificação 2: Limite de registro'] == 'Não classificado para troca') and (row['Classificação 3: Dimensionamento'] == 'Classificado para troca')
    c2_3 = (row['Classificação 1: Idade'] == 'Não classificado para troca') and (row['Classificação 2: Limite de registro'] == 'Classificado para troca') and (row['Classificação 3: Dimensionamento'] == 'Classificado para troca')
    c1_2_3 = (row['Classificação 1: Idade'] == 'Classificado para troca') and (row['Classificação 2: Limite de registro'] == 'Classificado para troca') and (row['Classificação 3: Dimensionamento'] == 'Classificado para troca')
    
    conditions = [semtrocar, c1, c2, c3, c1_2, c1_3, c2_3, c1_2_3]
    values = ['Sem troca - UCs não classificadas para troca', 'C1 - UCs classificadas para troca por idade, somente', 
              'C2 - UCs classificadas para troca por extrapolação do limite de registro, somente',
              'C3 - UCs classificadas para troca por dimensionamento, somente',
              'C1 e C2 - UCs classificadas para troca por idade e extrapolação de registro',
              'C1 e C3 - UCs classificadas para troca por idade e dimensionamento',
              'C2 e C3 - UCs classificadas para troca por limite de registro e dimensionamento',
              'Todos os critérios - UCs classificadas pelos três critérios']
    
    for i, condition in enumerate(conditions):
        if condition:
            return values[i]
    return np.nan

df_analise['Critério de troca'] = df_analise.apply(determinar_criterio_troca, axis=1)

#submedição estimada --------------------------------------------------------------------
df_analise['Submedição por idade (%)'] = np.where(df_analise['Idade do HD (anos)']*0.01>=0.2,0.2,df_analise['Idade do HD (anos)']*0.01)


#novo consumo estimado --------------------------------------------------------------------
df_analise['Consumo estimado (m³)'] = np.where(df_analise['Idade do HD (anos)']>=5,(df_analise['Consumo médio (m³/mes)']*df_analise['Submedição por idade (%)']) + df_analise['Consumo médio (m³/mes)'],np.nan)

#valor da fatura atual de água --------------------------------------------------------------------
df_analise['Fatura atual de água (R$/mês)'] = np.where(df_analise['Consumo médio (m³/mes)'].notna(),
                                          df_analise.apply(lambda row: calcular_fatura(row['Consumo médio (m³/mes)'], 
                                                                                    row['Quantidade de economias'], 
                                                                                    row['CATEGORIA_PRINCIPAL']), axis=1),
                                          np.nan)
df_analise['Fatura atual de água (R$/mês)'] = df_analise['Fatura atual de água (R$/mês)'].replace(0, np.nan).round(2)


#valor da fatura atual de esgoto --------------------------------------------------------------------
#df_analise[list(valor_esg.values())].fillna(0, inplace=True)
df_analise['Fatura atual de esgoto (R$/mês)'] = df_analise['Fatura atual de água (R$/mês)']*0.8
df_analise['Fatura atual de esgoto (R$/mês)'] = df_analise['Fatura atual de esgoto (R$/mês)'].round(2) 

#valor da fatura atual de água e esgoto juntas --------------------------------------------------------------------
df_analise['Fatura atual de água e esgoto (R$/mês)'] = np.add(df_analise['Fatura atual de água (R$/mês)'], df_analise['Fatura atual de esgoto (R$/mês)'])

#valor da fatura de agua estimada  --------------------------------------------------------------------
df_analise['Consumo médio (m³/mes)'].astype(float)
df_analise['Fatura ajustada de água (R$/mês)'] = df_analise.apply(lambda row: calcular_fatura(row['Consumo estimado (m³)'], row['Quantidade de economias'], row['CATEGORIA_PRINCIPAL']), axis=1).round(2)

#valor da fatura de esgoto estimada
df_analise['Fatura ajustada de esgoto (R$/mês)'] = np.where(df_analise['Fatura atual de esgoto (R$/mês)'] != 0, df_analise['Fatura ajustada de água (R$/mês)']*0.8, 0).round(2)

#valor da fatura total estimada
df_analise['Fatura ajustada de água e esgoto (R$/mês)'] = np.where(df_analise['Idade do HD (anos)']>=5,df_analise['Fatura ajustada de água (R$/mês)'] + df_analise['Fatura ajustada de esgoto (R$/mês)'],np.nan)


#ganho financeiro
df_analise['Ganho financeiro com ajustes (R$/ano)'] = np.where(df_analise['Idade do HD (anos)']>=5,(df_analise['Fatura ajustada de água e esgoto (R$/mês)'] - df_analise['Fatura atual de água e esgoto (R$/mês)'])*12,np.nan)

#tempo de retorno financeiro
df_analise['Tempo para retorno financeiro (meses)'] = np.where(df_analise['Ganho financeiro com ajustes (R$/ano)']>0,
                                          df_analise['Custo de troca (R$)']/(df_analise['Fatura ajustada de água e esgoto (R$/mês)']-df_analise['Fatura atual de água e esgoto (R$/mês)']),
                                         np.nan)
df_analise.drop(columns=['Fatura ajustada de água (R$/mês)','Fatura ajustada de esgoto (R$/mês)'], axis=1, inplace=True)
df_analise.drop(columns=['Fatura atual de água (R$/mês)','Fatura atual de esgoto (R$/mês)'], axis=1, inplace=True)


#Após cálculos, colocar porcentagens corretas
df_analise['Submedição por idade (%)'] = df_analise['Submedição por idade (%)'].map('{:.2%}'.format)

#Renomeando colunas
novo_nome_colunas = {
    matricula: 'Matrícula',
    bairro: 'Bairro',
    numero_hidrometro: 'Número do HD',
    marca_hidrometro: 'Marca do HD',
    categoria_principal: 'Categoria',
    data_instalacao_hidrometro: 'Data de instalação',
    capacidade_hidrometro: 'Capacidade do HD (Qmáx em m³/h)',
    diametro_hidrometro: 'Diâmetro do HD (mm)'
}

# Use um dicionário de compreensão para criar o dicionário com as variáveis como chaves
novo_nome_colunas = {variavel: novo_nome for variavel, novo_nome in novo_nome_colunas.items()}

# Use o método rename() para renomear as colunas
df_analise.rename(columns=novo_nome_colunas, inplace=True)

display(df_analise)

,Matrícula,Bairro,Número do HD,Marca do HD,Categoria,Data de instalação,Capacidade do HD (Qmáx em m³/h),Diâmetro do HD (mm),Quantidade de economias,Tipo de hidrômetro,Ano de instalação,Ano de fabricação,Idade do HD (anos),Consumo médio (m³/mes),Classificação 1: Idade,Registro estimado (m³),Registro máximo (m³),Classificação 2: Limite de registro,Modelo ideal,Capacidade ideal (m³/h),Diâmetro ideal (mm),Custo de troca (R$),Classificação 3: Dimensionamento,Critério de troca,Submedição por idade (%),Consumo estimado (m³),Fatura atual de água e esgoto (R$/mês),Fatura ajustada de água e esgoto (R$/mês),Ganho financeiro com ajustes (R$/ano),Tempo para retorno financeiro (meses)
0,1206122-0,Passa Vinte,A15S437045,Itron,Residencial,2015-10-22,1.5,3/4,1,Velocimétrico,2015.0,2015.0,8.12,8.916667,Classificado para troca,868.84,2900.0,Não classificado para troca,Tipo 1,1.5,3/4,222.0,Não classificado para troca,"C1 - UCs classificadas para troca por idade, s...",8.12%,9.6407,81.02,81.02,0.00,NaN
1,1206232-4,Pinheira,Y20HW0035553,Elster,Residencial,2021-08-17,1.5,3/4,1,Velocimétrico,2021.0,2020.0,2.30,1.166667,Não classificado para troca,32.2,2900.0,Não classificado para troca,Tipo 1,1.5,3/4,222.0,Não classificado para troca,Sem troca - UCs não classificadas para troca,2.30%,NaN,81.02,NaN,NaN,NaN
2,1206270-7,Aririú,Y18HW0113987,Elster,Residencial,2018-07-30,1.5,3/4,1,Velocimétrico,2018.0,2018.0,5.35,15.333333,Classificado para troca,984.4,2900.0,Não classificado para troca,Tipo 2,1.5,3/4,222.0,Não classificado para troca,"C1 - UCs classificadas para troca por idade, s...",5.35%,16.153667,160.65,172.91,147.12,18.107667
3,1206274-0,Pinheira,Y05X313742,Turbimax/Sensus,Residencial,2005-01-01,1.5,3/4,1,Velocimétrico,2005.0,2005.0,18.93,16.083333,Classificado para troca,3653.49,2900.0,Classificado para troca,Tipo 2,1.5,3/4,222.0,Não classificado para troca,C1 e C2 - UCs classificadas para troca por ida...,18.93%,19.127908,171.85,217.31,545.52,4.883414
4,1206275-8,Pinheira,Y05X313737,Turbimax/Sensus,Residencial,2005-01-01,1.5,3/4,1,Velocimétrico,2005.0,2005.0,18.93,1.333333,Classificado para troca,302.88,2900.0,Não classificado para troca,Tipo 1,1.5,3/4,222.0,Não classificado para troca,"C1 - UCs classificadas para troca por idade, s...",18.93%,1.585733,81.02,81.02,0.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58169,993917-2,Pinheira,Y18HW0114906,Elster,Residencial,2018-12-22,1.5,3/4,1,Velocimétrico,2018.0,2018.0,4.95,4.083333,Não classificado para troca,242.55,2900.0,Não classificado para troca,Tipo 1,1.5,3/4,222.0,Não classificado para troca,Sem troca - UCs não classificadas para troca,4.95%,NaN,81.02,NaN,NaN,NaN
58170,993922-9,Pinheira,Y20HW0036125,Elster,Residencial,2020-07-06,1.5,3/4,1,Velocimétrico,2020.0,2020.0,3.41,4.666667,Não classificado para troca,190.96,2900.0,Não classificado para troca,Tipo 1,1.5,3/4,222.0,Não classificado para troca,Sem troca - UCs não classificadas para troca,3.41%,NaN,81.02,NaN,NaN,NaN
58171,994366-8,Praia de Fora,Y18HW0114936,Elster,Residencial,2019-01-07,1.5,3/4,1,Velocimétrico,2019.0,2018.0,4.91,5.833333,Não classificado para troca,343.7,2900.0,Não classificado para troca,Tipo 1,1.5,3/4,222.0,Não classificado para troca,Sem troca - UCs não classificadas para troca,4.91%,NaN,81.02,NaN,NaN,NaN
58172,994367-6,Praia de Fora,Y12L341335,Lao,Residencial,2013-02-06,1.5,3/4,1,Velocimétrico,2013.0,2012.0,10.83,8.583333,Classificado para troca,1115.49,2900.0,Não classificado para troca,Tipo 1,1.5,3/4,222.0,Não classificado para troca,"C1 - UCs classificadas para troca por idade, s...",10.83%,9.512908,81.02,81.02,0.00,NaN


## Aqui eu exporto os dados análisados

In [8]:
#df_analise.to_excel('1.2. Dados tratados.xlsx')